# 01 — Does radiation change what the tumour is made of?

`09` closed the expression question: **radiation leaves no reproducible cell-intrinsic
signature in any compartment** of this dataset, while Cy + CAR-T leaves a strong one everywhere
through the identical pipeline. That result says nothing at all about the question asked here.

> **Does adding RT on top of Cy + CAR-T change the *composition* of the tumour — and does it
> change it in the tumour that was never in the beam?**

A compartment can double in abundance with every cell in it transcribing identically. Expression
and composition are orthogonal readouts, and outside the macrophage lineage (`07`) composition
has never been tested for RT in this study. It is also the readout that matches the shape of the
one RT effect that did survive (`08`/`08b`): **geometric, systemic, not about what cells say.**

### What this notebook can and cannot see
From `00`'s MDE table, the single-cell object supports composition claims about
`Classical_Mono`, `M1_like_Mac`, `Intermediate_Mac`, `N2_like_Neu` and `Cancer_cell` at effect
sizes below the known treated-vs-NoTx effect. It does **not** support them for `CD8_T`
(MDE ~4x the benchmark), `Treg`, `CD4_T` or `B` — those are reported for completeness and
explicitly not concluded from. **The effector arm is a spatial question** (`03`).

### The three ways this could produce a fake positive, and the control for each

| trap | why it bites here | control |
|---|---|---|
| **Closure** — proportions sum to 1 | treated samples are ~85% myeloid; a myeloid influx deflates every other fraction arithmetically, which reads as depletion | every contrast is run on **CLR** as well as raw proportion; where they disagree, the disagreement is reported as the finding |
| **The site effect** — `Tu1` differs from `Tu2` at baseline | the non-RT arms show the same `Tu1`/`Tu2` gap, so a within-arm `Tu1 − Tu2` is mostly anatomy | §C runs it as a **difference-in-differences** against the non-RT arms |
| **Multiplicity without independence** | 14 cell types on a simplex are not 14 independent tests; `09` got a parametric p = 1.7e-152 on a vector whose permutation p was 0.49 | §D's concordance statistic is scored against a **label permutation**, and benchmarked against treated-vs-NoTx through the same machinery |

### Design
`Tu1` = irradiated, `Tu2` = **abscopal**, same mouse. No RT-alone arm, so every contrast reads
*"adding RT on top of Cy + CAR-T"*. Two CAR-T backgrounds, n = 3 mice a side: the evidence is
**effect size plus sign-agreement across both backgrounds**, never a p-value.

## 1. Config

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, h5py, anndata as ad
import matplotlib.pyplot as plt
from scipy import stats
from math import comb

pd.set_option("display.width", 260); pd.set_option("display.max_columns", 60)

_HPC = Path("/coh_labs/yunroseli/Jona/CAR-T")
_LOC = Path("/Users/janzules/Roselab/Spatial/CAR_T")
PROJ = _HPC if _HPC.exists() else _LOC
SC_REF = (PROJ/"data/zarr/fullDataset/annotating_references/annotation_v4.h5ad"
          if PROJ is _HPC else PROJ/"data/sc-reference/annotation_v4.h5ad")
OUT = PROJ/"results/rt_advantage"; FIG = OUT/"figures"
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

# arm naming: single cell spells the non-specific arm CyTAG72, spatial spells it CyT72
SC2ARM = {"NoTx":"NoTx", "CyPSCA":"CyPSCA", "CyTAG72":"CyT72",
          "RTCyPSCA":"RTCyPSCA", "RTCyTAG72":"RTCyT72"}
CART_PAIRS = [("PSCA", "RTCyPSCA", "CyPSCA"), ("nonspecific", "RTCyT72", "CyT72")]
SITE = {1: "irradiated", 2: "abscopal"}

COARSE = {                       # the big-picture composition: 6 compartments, well powered
    "lymphoid":   ["CD8_T","CD4_T","Treg","NK","NKT","B"],
    "macrophage": ["M1_like_Mac","M2_like_Mac","Intermediate_Mac"],
    "monocyte":   ["Classical_Mono","Nonclassical_Mono"],
    "neutrophil": ["N1_like_Neu","N2_like_Neu"],
    "dendritic":  ["cDC","pDC"],
    "tumour":     ["Cancer_cell"],
    "stroma":     ["Endothelial","Fibroblast"],
}
DROP_FROM_TOTAL = ["Erythrocyte"]       # contaminant, not a TME compartment
MIN_MEDIAN  = 5        # a cell type needs this many cells in the median sample to be testable
PSEUDO      = 0.5
N_PERM      = 4000
SEED        = 0
rng = np.random.default_rng(SEED)
print(f"PROJ = {PROJ}\nOUT  = {OUT}")

## 2. Load and rebuild the design

Rebuilt from `file_name` rather than read from `00`, so this notebook stands alone (the kernel
runs on the HPC and cannot import local modules). `.X` is not touched at all — this notebook
counts cells, it does not measure expression.

In [ ]:
with h5py.File(SC_REF, "r") as f:
    o = ad.io.read_elem(f["obs"])
print(f"{len(o):,} cells")

o["sample"] = o["file_name"].astype(str)
p = o["sample"].str.extract(r"^(?P<sc>.+)_Tu(?P<site>\d)_(?P<rep>\d+)$")
assert p.notna().all().all(), "file_name did not parse"
o["arm"]   = p["sc"].map(SC2ARM)
o["site"]  = p["site"].astype(int)
o["mouse"] = p["sc"] + "_m" + p["rep"]
o["ct"]    = o["cell_type_lvl2"].astype(str)
assert o["arm"].notna().all(), f"unmapped arms: {set(p.loc[o.arm.isna(),'sc'])}"

design = (o.groupby("sample", observed=True)
            .agg(arm=("arm","first"), site=("site","first"), mouse=("mouse","first"),
                 n_cells=("arm","size")))
design["RT"]   = design.arm.str.startswith("RT")
design["cart"] = design.arm.map({"CyPSCA":"PSCA","RTCyPSCA":"PSCA",
                                 "CyT72":"nonspecific","RTCyT72":"nonspecific","NoTx":"none"})
print("\nsamples per arm x site:")
print(design.pivot_table(index="arm", columns="site", values="n_cells", aggfunc="count")
        .rename(columns=SITE).fillna(0).astype(int).to_string())

## 3. Two compositions, and the CLR transform

**Coarse** (7 compartments) is the big-picture readout and is the one with real power.
**Fine** (individual `cell_type_lvl2`) has resolution but only for the types `00` cleared.

CLR = `log(p_i) − mean_j log(p_j)`: each compartment read against the sample's own geometric
mean instead of against a fixed total. This is what stops a myeloid influx from registering as
depletion of everything else. Its one cost is that the CLR vector sums to zero across
compartments by construction — the features are not independent, which is exactly why §D scores
against a permutation rather than a parametric p.

In [ ]:
def clr(counts, pseudo=PSEUDO):
    x = counts.values.astype(float) + pseudo
    lg = np.log(x / x.sum(1, keepdims=True))
    return pd.DataFrame(lg - lg.mean(1, keepdims=True), index=counts.index, columns=counts.columns)

fine_all = pd.crosstab(o["sample"], o["ct"]).drop(columns=DROP_FROM_TOTAL, errors="ignore")
fine_all = fine_all.reindex(design.index)

coarse = pd.DataFrame({k: fine_all[[c for c in v if c in fine_all]].sum(1)
                       for k, v in COARSE.items()}, index=fine_all.index)
assigned = sum(len([c for c in v if c in fine_all]) for v in COARSE.values())
print(f"coarse compartments: {list(coarse.columns)}  "
      f"({assigned}/{fine_all.shape[1]} cell types assigned)")
unassigned = [c for c in fine_all.columns if not any(c in v for v in COARSE.values())]
print(f"unassigned (excluded): {unassigned}")

FINE = fine_all.columns[fine_all.median(0) >= MIN_MEDIAN].tolist()
print(f"\nfine basis: {len(FINE)}/{fine_all.shape[1]} types with median >= {MIN_MEDIAN}/sample")
print(f"  tested   : {FINE}")
print(f"  too rare : {[c for c in fine_all.columns if c not in FINE]}")

P_coarse = coarse.div(coarse.sum(1), axis=0)
P_fine   = fine_all[FINE].div(fine_all[FINE].sum(1), axis=0)
Zc = clr(coarse).join(design[["arm","site","mouse","RT","cart"]])
Zf = clr(fine_all[FINE]).join(design[["arm","site","mouse","RT","cart"]])

print("\nmean composition by arm x site (% of assigned cells):")
print((P_coarse.join(design[["arm","site"]]).groupby(["arm","site"], observed=True).mean()*100)
        .round(1).to_string())
coarse.to_csv(OUT/"composition_counts_coarse.csv")
fine_all.to_csv(OUT/"composition_counts_fine.csv")

## 4. §A — Positive control: does the pipeline see a perturbation it is known to contain?

Treated (Cy + CAR-T, any background) vs `NoTx`. `07` and `00` both say this is enormous. If it
does not come out enormous here, the composition machinery is broken and nothing below means
anything. This also sets the **yardstick**: the RT effect in §B is reported as a fraction of
this.

In [ ]:
def delta(Z, feats, a_mask, b_mask):
    return Z.loc[a_mask, feats].mean() - Z.loc[b_mask, feats].mean()

pos_c = delta(Zc, coarse.columns, (Zc.arm != "NoTx").values, (Zc.arm == "NoTx").values)
pos_f = delta(Zf, FINE,           (Zf.arm != "NoTx").values, (Zf.arm == "NoTx").values)

pc = pd.DataFrame({
    "CLR_delta": pos_c,
    "pct_NoTx":  (P_coarse[Zc.arm == "NoTx"].mean()*100),
    "pct_treated": (P_coarse[Zc.arm != "NoTx"].mean()*100),
})
pc["log2_prop_FC"] = np.log2(pc.pct_treated / pc.pct_NoTx)
print("COARSE — treated vs NoTx (the yardstick)")
print(pc.sort_values("CLR_delta", ascending=False).round(3).to_string())

print("\nFINE — treated vs NoTx")
print(pos_f.sort_values(ascending=False).round(3).to_string())

BENCH_c, BENCH_f = pos_c.abs().mean(), pos_f.abs().mean()
print(f"\nyardstick |CLR delta|: coarse {BENCH_c:.3f}, fine {BENCH_f:.3f}")
print("A real perturbation in this assay moves compartments by this much on the log scale.")

## 5. §B — The RT contrast, site-matched, in both CAR-T backgrounds

`RTCy* − Cy*`, computed separately within `Tu1` (irradiated) and `Tu2` (abscopal) so the site is
never confounded with the beam. Run once per CAR-T background. At n = 3 a side the claim is
**sign agreement between the two backgrounds**, and the size relative to §A's yardstick.

Raw-proportion log2FC is printed beside the CLR delta on purpose. **Where the two disagree in
sign, the apparent change is closure**, not biology.

In [ ]:
def rt_table(Z, P, feats, site=None):
    rows = []
    for bg, rt_arm, ct_arm in CART_PAIRS:
        m_site = np.ones(len(Z), bool) if site is None else (Z.site == site).values
        a = (Z.arm == rt_arm).values & m_site
        b = (Z.arm == ct_arm).values & m_site
        if a.sum() < 2 or b.sum() < 2:
            continue
        d_clr  = delta(Z, feats, a, b)
        d_prop = np.log2(P.loc[a, feats].mean() / P.loc[b, feats].mean())
        for f_ in feats:
            rows.append(dict(background=bg, site=site, feature=f_, n_rt=int(a.sum()),
                             n_ctrl=int(b.sum()), clr_delta=d_clr[f_],
                             log2_prop_FC=d_prop[f_]))
    return pd.DataFrame(rows)


def widen(tab, bench):
    w = tab.pivot_table(index="feature", columns="background",
                        values=["clr_delta", "log2_prop_FC"])
    w.columns = [f"{a}_{b}" for a, b in w.columns]
    cl = [c for c in w.columns if c.startswith("clr_delta")]
    pr = [c for c in w.columns if c.startswith("log2_prop")]
    w["clr_mean"] = w[cl].mean(1)
    w["sign_agree"] = np.where(np.sign(w[cl[0]]) == np.sign(w[cl[1]]),
                               np.where(w[cl[0]] > 0, "UP both", "DOWN both"), "disagree")
    w["closure_flag"] = np.where(np.sign(w["clr_mean"]) != np.sign(w[pr].mean(1)),
                                 "CLR vs prop DISAGREE", "")
    w["frac_of_yardstick"] = w["clr_mean"].abs() / bench
    return w.sort_values("clr_mean", ascending=False)


res = {}
for lbl, Z, P, feats, bench in [("coarse", Zc, P_coarse, list(coarse.columns), BENCH_c),
                                ("fine",   Zf, P_fine,   FINE,                BENCH_f)]:
    for site in [1, 2]:
        t = rt_table(Z, P, feats, site=site)
        w = widen(t, bench)
        res[(lbl, site)] = w
        print(f"\n{'='*92}\n{lbl.upper()} — RT vs matched non-RT, {SITE[site]} tumour (Tu{site})\n{'='*92}")
        print(w.round(3).to_string())
        n_ag = int((w.sign_agree != "disagree").sum())
        print(f"  sign-agreeing features: {n_ag}/{len(w)} "
              f"(chance ~{len(w)/2:.0f})   largest |effect| as fraction of the "
              f"treated-vs-NoTx yardstick: {w.frac_of_yardstick.max():.2f}")

pd.concat({k: v for k, v in res.items()}, names=["scale", "site"]).to_csv(
    OUT/"rt_composition_contrasts.csv")

## 6. §C — Within-mouse difference-in-differences: is any of it the beam?

`Tu1 − Tu2` inside an RT arm is a within-animal contrast — same mouse, same batch, same systemic
therapy, one tumour in the beam and one not. It is also **mostly anatomy**: the non-RT arms show
a `Tu1`/`Tu2` difference too. So the quantity that isolates the beam is

```
DiD = (Tu1 − Tu2)_RT-arm  −  (Tu1 − Tu2)_matched-non-RT-arm
```

The non-RT `Tu1 − Tu2` column is printed alongside. **If it is as large as the RT one, the
finding is the tumour site, not the radiation** — which is exactly what happened to the
fatty-acid claim in `09`.

In [ ]:
def within_mouse(Z, feats, arm):
    """Per-mouse Tu1 - Tu2. Mice missing either site are dropped, not carried as NaN — a
    half-present mouse would otherwise be counted in n and shuffled by the permutation in §D
    while contributing nothing to the mean."""
    d = Z[Z.arm == arm]
    w = d.reset_index().pivot_table(index="mouse", columns="site", values=feats)
    out = {}
    for f_ in feats:
        if 1 in w[f_].columns and 2 in w[f_].columns:
            out[f_] = (w[f_][1] - w[f_][2])
    return pd.DataFrame(out).dropna(how="all")          # mice x features


did_rows = []
for lbl, Z, feats in [("coarse", Zc, list(coarse.columns)), ("fine", Zf, FINE)]:
    for bg, rt_arm, ct_arm in CART_PAIRS:
        a, b = within_mouse(Z, feats, rt_arm), within_mouse(Z, feats, ct_arm)
        if a.empty or b.empty:
            continue
        for f_ in feats:
            did_rows.append(dict(scale=lbl, background=bg, feature=f_,
                                 n_mice_rt=int(a[f_].notna().sum()),
                                 n_mice_ctrl=int(b[f_].notna().sum()),
                                 tu1_minus_tu2_RT=a[f_].mean(),
                                 tu1_minus_tu2_nonRT=b[f_].mean(),
                                 DiD=a[f_].mean() - b[f_].mean()))
did = pd.DataFrame(did_rows)
did.to_csv(OUT/"rt_composition_did.csv", index=False)

for lbl in ["coarse", "fine"]:
    d = did[did.scale == lbl]
    w = d.pivot_table(index="feature", columns="background",
                      values=["tu1_minus_tu2_RT", "tu1_minus_tu2_nonRT", "DiD"])
    w.columns = [f"{a}_{b}" for a, b in w.columns]
    dcols = [c for c in w.columns if c.startswith("DiD")]
    w["DiD_mean"] = w[dcols].mean(1)
    w["DiD_sign_agree"] = np.sign(w[dcols[0]]) == np.sign(w[dcols[1]])
    print(f"\n{'='*92}\n{lbl.upper()} — within-mouse difference-in-differences (Tu1 vs Tu2)"
          f"\n{'='*92}")
    print(w.sort_values("DiD_mean", ascending=False).round(3).to_string())
    site_only = w[[c for c in w.columns if c.startswith("tu1_minus_tu2_nonRT")]].abs().mean().mean()
    beam_only = w[dcols].abs().mean().mean()
    print(f"  mean |site effect| (non-RT arms)      : {site_only:.3f}")
    print(f"  mean |DiD| (what the beam adds to it) : {beam_only:.3f}")
    print(f"  -> the beam is {beam_only/site_only:.2f}x the pure site effect"
          + ("  (site dominates — do not report Tu1-vs-Tu2 as an RT effect)"
             if beam_only < site_only else ""))

## 7. §D — The decisive test: does the RT effect reproduce across the two CAR-T backgrounds?

The same logic as `09 §D`, moved from genes to compartments. A real RT effect must produce the
**same compositional shift in both CAR-T backgrounds**. Two statistics, because at 7 or 14
features neither alone is enough:

* **Pearson r** between the two backgrounds' CLR-delta vectors;
* **sign agreement count** — more robust than r when the vector is short.

Both are scored against a **label permutation** that shuffles RT / non-RT among the mice *within
each background and site*, preserving the design. And both are run on **treated-vs-NoTx** through
the identical machinery, which is what a real effect scores here.

The same pair of statistics is also run on **§C's within-mouse DiD vector** — that contrast
removes animal-to-animal variance, so it can show structure the between-animal contrast cannot,
and until now it had no null at all. A `RT within-mouse DiD` row that clears its permutation
while the between-arm rows do not would mean the RT effect is real but swamped by inter-animal
spread.

The CLR vector sums to zero across features, so even the null vectors are negatively correlated
on average — which is precisely why the observed r is meaningless except against this null.

In [ ]:
def delta_vec(Z, feats, rt_arm, ct_arm, is_rt=None, sites=(1, 2)):
    d = Z[Z.arm.isin([rt_arm, ct_arm])]
    lab = (d.index.map(is_rt).values.astype(bool) if is_rt is not None
           else (d.arm == rt_arm).values)
    parts = []
    for s in sites:
        m = (d.site == s).values
        if (m & lab).sum() < 2 or (m & ~lab).sum() < 2:
            continue
        parts.append(d.loc[m & lab, feats].mean() - d.loc[m & ~lab, feats].mean())
    return pd.concat(parts, axis=1).mean(axis=1) if parts else None


def concordance(Z, feats, pairs, n_perm=N_PERM, sites=(1, 2), seed=SEED):
    rg = np.random.default_rng(seed)

    def stat(is_rt):
        v = [delta_vec(Z, feats, r, c, is_rt, sites) for _, r, c in pairs]
        if any(x is None for x in v):
            return np.nan, np.nan
        a, b = v[0].values, v[1].values
        r_ = stats.pearsonr(a, b)[0] if np.std(a) > 0 and np.std(b) > 0 else np.nan
        return r_, float((np.sign(a) == np.sign(b)).mean())

    obs_r, obs_s = stat(None)
    nr, ns = [], []
    for _ in range(n_perm):
        is_rt = {}
        for _bg, rt_arm, ct_arm in pairs:
            for s in sites:
                idx = Z.index[(Z.arm.isin([rt_arm, ct_arm])) & (Z.site == s)]
                k = int((Z.loc[idx, "arm"] == rt_arm).sum())
                pm = rg.permutation(len(idx))
                is_rt.update({i: bool(pm[j] < k) for j, i in enumerate(idx)})
        r_, s_ = stat(is_rt)
        nr.append(r_); ns.append(s_)
    nr, ns = np.array(nr, float), np.array(ns, float)
    return dict(r=obs_r, r_null_mean=np.nanmean(nr),
                p_r=float(np.nanmean(nr >= obs_r)),
                sign_frac=obs_s, sign_null_mean=np.nanmean(ns),
                p_sign=float(np.nanmean(ns >= obs_s)))


# the benchmark: treated-vs-NoTx, same machinery. "Backgrounds" become the two CAR-T arms,
# each against the shared NoTx baseline.
BENCH_PAIRS = [("PSCA", "CyPSCA", "NoTx"), ("nonspecific", "CyT72", "NoTx")]

def did_concordance(Z, feats, pairs, n_perm=N_PERM, seed=SEED):
    """Same statistic as `concordance`, but on the WITHIN-MOUSE DiD vector.

    One Tu1-Tu2 vector per mouse; the permutation shuffles which mice are the RT ones,
    inside each CAR-T background. This is the null §C lacked.
    """
    rg = np.random.default_rng(seed + 1)
    blocks = {}
    for bg, rt_arm, ct_arm in pairs:
        a, b = within_mouse(Z, feats, rt_arm), within_mouse(Z, feats, ct_arm)
        if a.empty or b.empty:
            return dict(r=np.nan, r_null_mean=np.nan, p_r=np.nan,
                        sign_frac=np.nan, sign_null_mean=np.nan, p_sign=np.nan)
        blocks[bg] = (pd.concat([a, b]), np.r_[np.ones(len(a), bool), np.zeros(len(b), bool)])

    def stat(perm):
        v = []
        for bg, _r, _c in pairs:
            W, lab = blocks[bg]
            lab = lab if perm is None else perm[bg]
            v.append(W[lab].mean() - W[~lab].mean())
        a, b = v[0].values, v[1].values
        r_ = stats.pearsonr(a, b)[0] if np.std(a) > 0 and np.std(b) > 0 else np.nan
        return r_, float((np.sign(a) == np.sign(b)).mean())

    obs_r, obs_s = stat(None)

    # leverage: a correlation over 7-14 points can be carried by one compartment. Drop each
    # feature in turn and report the worst surviving r, plus which feature was doing the work.
    vv = []
    for bg, _r, _c in pairs:
        W, lab = blocks[bg]
        vv.append((W[lab].mean() - W[~lab].mean()))
    A, B = vv[0], vv[1]
    jack = {f_: stats.pearsonr(A.drop(f_).values, B.drop(f_).values)[0] for f_ in A.index}
    j_min_f = min(jack, key=jack.get)

    nr, ns = [], []
    for _ in range(n_perm):
        perm = {bg: rg.permutation(lab) for bg, (W, lab) in blocks.items()}
        r_, s_ = stat(perm)
        nr.append(r_); ns.append(s_)
    nr, ns = np.array(nr, float), np.array(ns, float)
    # n mice is small, so the permutation has a hard resolution floor: with C(n, k) distinct
    # relabellings per background there are only prod(...) distinct null draws, and no p below
    # 1/that is achievable however many resamples are taken. Report it next to the p.
    n_distinct = int(np.prod([comb(len(lab), int(lab.sum())) for _W, lab in blocks.values()]))
    return dict(r=obs_r, r_null_mean=np.nanmean(nr), p_r=float(np.nanmean(nr >= obs_r)),
                sign_frac=obs_s, sign_null_mean=np.nanmean(ns),
                p_sign=float(np.nanmean(ns >= obs_s)),
                n_mice=" vs ".join(f"{int(lab.sum())}v{int((~lab).sum())}"
                                   for _W, lab in blocks.values()),
                n_distinct_perms=n_distinct, p_floor=1.0/n_distinct,
                r_jack_min=jack[j_min_f], r_jack_worst_feature=j_min_f)


rows = []
for lbl, Z, feats in [("coarse", Zc, list(coarse.columns)), ("fine", Zf, FINE)]:
    for name, pairs, sites in [
            ("RT effect (both sites)",        CART_PAIRS,  (1, 2)),
            ("RT effect (irradiated only)",   CART_PAIRS,  (1,)),
            ("RT effect (ABSCOPAL only)",     CART_PAIRS,  (2,)),
            ("BENCHMARK treated vs NoTx",     BENCH_PAIRS, (1, 2))]:
        r = concordance(Z, feats, pairs, sites=sites)
        rows.append(dict(scale=lbl, contrast=name, n_features=len(feats), **r))
    rows.append(dict(scale=lbl, contrast="RT within-mouse DiD", n_features=len(feats),
                     **did_concordance(Z, feats, CART_PAIRS)))
    # and again with the sparse compartments dropped — stroma / Fibroblast / Endothelial sit at
    # 5-6 cells in the median sample, and they are the biggest points on the DiD scatter
    solid = [f_ for f_ in feats if (coarse if lbl == "coarse" else fine_all)[
                 [f_] if lbl == "coarse" else [f_]].median().iloc[0] >= 30]
    if 3 <= len(solid) < len(feats):
        rows.append(dict(scale=lbl, contrast="RT DiD, sparse compartments dropped",
                         n_features=len(solid),
                         **did_concordance(Z, solid, CART_PAIRS)))
conc = pd.DataFrame(rows)
conc.to_csv(OUT/"rt_composition_concordance.csv", index=False)
print("CROSS-BACKGROUND CONCORDANCE  (p from label permutation, one-sided)")
print(conc.round(3).to_string(index=False))
print("\nreading: an RT row is only interesting if its r and sign_frac sit clearly above their")
print("own null AND are in the neighbourhood of the BENCHMARK row on the same scale.")
print("\nr_jack_min is the correlation after dropping the single most influential compartment.")
print("If it collapses, the concordance is one feature wearing a trenchcoat — check which one in")
print("r_jack_worst_feature, and check whether that compartment is above the count floor.")
print("\np_floor is the smallest p the design can produce: with n = 2-3 mice a side there are")
print("only a few hundred distinct relabellings, so a p AT the floor means 'the observed value")
print("beat every relabelling', not 'p = 0.005 in the usual sense'. Read it as a rank, not a rate.")

## 8. §E — The abscopal reading, and the focal ratios

The abscopal tumour is the point of the experiment, so it gets its own summary rather than being
one column of §B. Two questions:

1. **Is there a compositional RT effect in `Tu2` at all?** (§D's abscopal-only row)
2. **Is it smaller than at the irradiated site?** If abscopal ≈ irradiated, the mechanism is
   systemic — which is what `08b` found for the range effect. If abscopal < irradiated, it is
   dose-driven.

Then the pre-specified ratios, which are closure-proof by construction (a ratio of two
compartments does not care what the rest of the sample did):
`M1/M2`, `monocyte/macrophage` (the recruitment-to-differentiation axis), `lymphoid/myeloid`,
`CD8/Treg`, and `tumour/immune`.

In [ ]:
ab = conc[conc.contrast.str.contains("ABSCOPAL")].set_index("scale")
ir = conc[conc.contrast.str.contains("irradiated")].set_index("scale")
print("abscopal vs irradiated, same statistic:")
for lbl in ["coarse", "fine"]:
    print(f"  {lbl:<7s} irradiated r={ir.loc[lbl,'r']:+.3f} (p={ir.loc[lbl,'p_r']:.3f})   "
          f"abscopal r={ab.loc[lbl,'r']:+.3f} (p={ab.loc[lbl,'p_r']:.3f})")

RATIOS = {
    "M1_over_M2":        (["M1_like_Mac"], ["M2_like_Mac"]),
    "mono_over_mac":     (["Classical_Mono","Nonclassical_Mono"],
                          ["M1_like_Mac","M2_like_Mac","Intermediate_Mac"]),
    "lymphoid_over_myeloid": (["CD8_T","CD4_T","Treg","NK","NKT","B"],
                          ["M1_like_Mac","M2_like_Mac","Intermediate_Mac","Classical_Mono",
                           "Nonclassical_Mono","N1_like_Neu","N2_like_Neu","cDC","pDC"]),
    "CD8_over_Treg":     (["CD8_T"], ["Treg"]),
    "tumour_over_immune":(["Cancer_cell"],
                          ["CD8_T","CD4_T","Treg","NK","NKT","B","M1_like_Mac","M2_like_Mac",
                           "Intermediate_Mac","Classical_Mono","Nonclassical_Mono",
                           "N1_like_Neu","N2_like_Neu","cDC","pDC"]),
}
R = pd.DataFrame({k: np.log2((fine_all[[c for c in a if c in fine_all]].sum(1) + PSEUDO) /
                             (fine_all[[c for c in b if c in fine_all]].sum(1) + PSEUDO))
                  for k, (a, b) in RATIOS.items()}).join(design[["arm","site","mouse"]])

TOO_RARE = [c for c in fine_all.columns if c not in FINE]
frag = {k: sorted(set(a + b) & set(TOO_RARE)) for k, (a, b) in RATIOS.items()}
print("count-floor check — a ratio is only as good as its rarest component:")
for k, bad_ in frag.items():
    med = {c: int(fine_all[c].median()) for c in RATIOS[k][0] + RATIOS[k][1] if c in fine_all}
    worst = min(med, key=med.get)
    print(f"  {k:<22s} rarest component {worst} at {med[worst]} cells in the median sample"
          + (f"   <-- FRAGILE, below the {MIN_MEDIAN}-cell floor: {bad_}" if bad_ else ""))

print("\nlog2 ratios, mean per arm x site:")
print(R.groupby(["arm","site"], observed=True)[list(RATIOS)].mean().round(2).to_string())

rows = []
for k in RATIOS:
    for bg, rt_arm, ct_arm in CART_PAIRS:
        for site in [1, 2]:
            a = R[(R.arm == rt_arm) & (R.site == site)][k]
            b = R[(R.arm == ct_arm) & (R.site == site)][k]
            if len(a) < 2 or len(b) < 2:
                continue
            rows.append(dict(ratio=k, background=bg, site=SITE[site],
                             n_rt=len(a), n_ctrl=len(b),
                             delta_log2=a.mean()-b.mean(),
                             rt_mean=a.mean(), ctrl_mean=b.mean(),
                             separated=bool(a.min() > b.max() or a.max() < b.min())))
Rt = pd.DataFrame(rows)
w = Rt.pivot_table(index=["ratio","site"], columns="background", values="delta_log2")
w["mean"] = w.mean(1)
w["sign_agree"] = np.sign(w.iloc[:, 0]) == np.sign(w.iloc[:, 1])
w["fragile"] = [", ".join(frag[r]) or "" for r, _s in w.index]
w["fully_separated_in"] = [
    ", ".join(Rt[(Rt.ratio == r) & (Rt.site == s) & Rt.separated].background) or "-"
    for r, s in w.index]
print("\nFOCAL RATIOS — RT minus matched non-RT (log2), by site")
print(w.round(3).to_string())
Rt.to_csv(OUT/"rt_focal_ratios.csv", index=False); R.to_csv(OUT/"focal_ratios_per_sample.csv")

## 9. Figures

In [ ]:
ORDER = ["NoTx","CyPSCA","RTCyPSCA","CyT72","RTCyT72"]
pal = plt.get_cmap("tab10")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), sharey=True)
for ax, site in zip(axes, [1, 2]):
    d = (P_coarse.join(design[["arm","site"]]).query("site == @site")
         .groupby("arm", observed=True).mean(numeric_only=True).reindex(ORDER).dropna(how="all"))
    bottom = np.zeros(len(d))
    for i, c in enumerate(coarse.columns):
        ax.bar(d.index, d[c]*100, bottom=bottom*100, label=c, color=pal(i % 10))
        bottom += d[c].values
    ax.set_title(f"{SITE[site]} tumour (Tu{site})"); ax.set_ylabel("% of assigned cells")
    ax.tick_params(axis="x", rotation=30)
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
fig.suptitle("Compartment composition by arm — single cell", fontsize=13)
fig.tight_layout(); fig.savefig(FIG/"01_composition_stacked.png", dpi=250, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex="col")
for row, (lbl, feats, bench) in enumerate([("coarse", list(coarse.columns), BENCH_c),
                                           ("fine", FINE, BENCH_f)]):
    for col, site in enumerate([1, 2]):
        ax = axes[row, col]
        w = res[(lbl, site)]
        y = np.arange(len(w))
        cl = [c for c in w.columns if c.startswith("clr_delta")]
        ax.axvline(0, color="k", lw=0.8)
        for span, c in [(bench, "#bbbbbb")]:
            ax.axvspan(-span, span, color=c, alpha=0.35, zorder=0)
        ax.scatter(w[cl[0]], y-0.13, s=48, color="#3b6ea5", label=cl[0].split("_")[-1])
        ax.scatter(w[cl[1]], y+0.13, s=48, color="#d1495b", label=cl[1].split("_")[-1])
        for yy, (_, r) in zip(y, w.iterrows()):
            ax.plot([r[cl[0]], r[cl[1]]], [yy-0.13, yy+0.13],
                    color=("#2a9d8f" if r.sign_agree != "disagree" else "#cccccc"),
                    lw=2.2, zorder=0)
        ax.set_yticks(y); ax.set_yticklabels(w.index, fontsize=9)
        ax.set_title(f"{lbl} — {SITE[site]} (Tu{site})", fontsize=11)
        if row == 1: ax.set_xlabel("CLR delta,  RT minus matched non-RT")
        if col == 0 and row == 0:
            ax.legend(fontsize=9, title="CAR-T background", loc="best")
fig.suptitle("RT effect on composition. Grey band = mean |treated-vs-NoTx| effect (the yardstick).\n"
             "Green connector = the two backgrounds agree in sign.", fontsize=12)
fig.tight_layout(); fig.savefig(FIG/"01_rt_composition_forest.png", dpi=250, bbox_inches="tight")
plt.show()

In [ ]:
# The money plot for §D: if the DiD were noise these two axes would be uncorrelated.
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.3))
for ax, (lbl, feats) in zip(axes[:2], [("coarse", list(coarse.columns)), ("fine", FINE)]):
    d = did[did.scale == lbl].pivot_table(index="feature", columns="background", values="DiD")
    r_ = conc.query("scale == @lbl and contrast == 'RT within-mouse DiD'").iloc[0]
    lim = np.abs(d.values).max()*1.25
    ax.axhline(0, color="k", lw=.8); ax.axvline(0, color="k", lw=.8)
    ax.plot([-lim, lim], [-lim, lim], ls="--", color="#bbbbbb", lw=1)
    ax.scatter(d["PSCA"], d["nonspecific"], s=60, color="#2a9d8f", zorder=3)
    for f_, row in d.iterrows():
        ax.annotate(f_, (row["PSCA"], row["nonspecific"]), fontsize=8,
                    xytext=(4, 4), textcoords="offset points")
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.set_xlabel("DiD, PSCA background"); ax.set_ylabel("DiD, non-specific background")
    ax.set_title(f"{lbl}: r = {r_.r:+.3f}  (null {r_.r_null_mean:+.3f}, p = {r_.p_r:.3f},\n"
                 f"floor {r_.p_floor:.3f} over {int(r_.n_distinct_perms)} relabellings)",
                 fontsize=10)

# the null distribution the observed r is being judged against
ax = axes[2]
rg_ = np.random.default_rng(SEED + 99)
nulls = []
blocks = {bg: (pd.concat([within_mouse(Zc, list(coarse.columns), rt),
                          within_mouse(Zc, list(coarse.columns), ct)]),
               np.r_[np.ones(len(within_mouse(Zc, list(coarse.columns), rt)), bool),
                     np.zeros(len(within_mouse(Zc, list(coarse.columns), ct)), bool)])
          for bg, rt, ct in CART_PAIRS}
for _ in range(2000):
    v = []
    for bg, _r, _c in CART_PAIRS:
        W, lab = blocks[bg]; lab = rg_.permutation(lab)
        v.append((W[lab].mean() - W[~lab].mean()).values)
    nulls.append(stats.pearsonr(v[0], v[1])[0])
obs = conc.query("scale == 'coarse' and contrast == 'RT within-mouse DiD'").iloc[0].r
ax.hist(nulls, bins=40, color="#cccccc", edgecolor="white")
ax.axvline(obs, color="#d1495b", lw=2.5, label=f"observed r = {obs:+.3f}")
ax.set_xlabel("cross-background r under label permutation"); ax.set_ylabel("permutations")
ax.set_title("coarse DiD — observed vs its own null", fontsize=10); ax.legend(fontsize=9)

fig.suptitle("§D — does the within-mouse RT effect reproduce in the second CAR-T background?",
             fontsize=13)
fig.tight_layout(); fig.savefig(FIG/"01_did_concordance.png", dpi=250, bbox_inches="tight")
plt.show()

In [ ]:
ks = list(RATIOS)
fig, axes = plt.subplots(2, len(ks), figsize=(3.1*len(ks), 7.4), sharey="col")
for row, site in enumerate([1, 2]):
    for col, k in enumerate(ks):
        ax = axes[row, col]
        d = R[R.site == site]
        for i, arm in enumerate(ORDER):
            v = d[d.arm == arm][k].dropna()
            if not len(v): continue
            ax.scatter(np.full(len(v), i) + rng.uniform(-.09, .09, len(v)), v, s=34,
                       color=("#d1495b" if arm.startswith("RT") else "#3b6ea5"
                              if arm != "NoTx" else "#999999"), zorder=3)
            ax.plot([i-.22, i+.22], [v.mean()]*2, color="k", lw=2)
        ax.set_xticks(range(len(ORDER)))
        ax.set_xticklabels(ORDER, rotation=90, fontsize=8)
        if col == 0: ax.set_ylabel(f"{SITE[site]}\nlog2 ratio")
        if row == 0: ax.set_title(k, fontsize=10)
        ax.grid(axis="y", alpha=.3)
fig.suptitle("Focal ratios, one dot per mouse. Red = RT arms.", fontsize=12)
fig.tight_layout(); fig.savefig(FIG/"01_focal_ratios.png", dpi=250, bbox_inches="tight")
plt.show()

## 10. Summary

In [ ]:
print("="*96); print("SUMMARY — report these back"); print("="*96)

print("\n[positive control — treated vs NoTx]")
top = pos_c.abs().sort_values(ascending=False).head(4)
print(f"  mean |CLR delta| coarse {BENCH_c:.3f} | fine {BENCH_f:.3f}")
print("  biggest movers: " + ", ".join(f"{k} {pos_c[k]:+.2f}" for k in top.index))
print(f"  -> pipeline {'SEES the known perturbation' if BENCH_c > 0.3 else 'FAILED the positive control — stop here'}")

print("\n[§B RT contrast, sign agreement across the two CAR-T backgrounds]")
for (lbl, site), w in res.items():
    ag = w[w.sign_agree != "disagree"]
    big = ag[ag.frac_of_yardstick > 0.25].sort_values("clr_mean", key=abs, ascending=False)
    print(f"  {lbl:<7s} {SITE[site]:<11s} {len(ag)}/{len(w)} agree; "
          f"of those, >25% of yardstick: "
          + (", ".join(f"{i} {r.clr_mean:+.2f} ({r.sign_agree})" for i, r in big.head(4).iterrows())
             or "none"))
    cf = w[w.closure_flag != ""].index.tolist()
    if cf: print(f"          closure warning (CLR and proportion disagree in sign): {cf}")

print("\n[§C within-mouse DiD]")
for lbl in ["coarse", "fine"]:
    d = did[did.scale == lbl]
    s_ = d.tu1_minus_tu2_nonRT.abs().mean(); b_ = d.DiD.abs().mean()
    print(f"  {lbl:<7s} mean |site effect| {s_:.3f} vs mean |DiD| {b_:.3f}  "
          f"-> beam/site = {b_/s_:.2f}")

print("\n[§D cross-background concordance — the decisive test]")
for _, r in conc.iterrows():
    flag = "  <-- BENCHMARK" if "BENCHMARK" in r.contrast else ""
    if np.isfinite(r.get("p_floor", np.nan)) and r.p_r <= 1.2*r.p_floor:
        flag += f"  <-- AT THE DESIGN CEILING: beat all {int(r.n_distinct_perms)} distinct relabellings"
    print(f"  {r.scale:<7s} {r.contrast:<30s} r={r.r:+.3f} (null {r.r_null_mean:+.3f}, "
          f"p={r.p_r:.3f})  sign={r.sign_frac:.2f} (null {r.sign_null_mean:.2f}, "
          f"p={r.p_sign:.3f}){flag}")

print("\n[§E abscopal]")
for lbl in ["coarse", "fine"]:
    print(f"  {lbl:<7s} irradiated r={ir.loc[lbl,'r']:+.3f} p={ir.loc[lbl,'p_r']:.3f} | "
          f"abscopal r={ab.loc[lbl,'r']:+.3f} p={ab.loc[lbl,'p_r']:.3f}")
agree_ratios = [f"{r} @ {s}: {v:+.2f}" for (r, s), v in
                (Rt.pivot_table(index=["ratio","site"], columns="background",
                                values="delta_log2").pipe(
                    lambda t: t[np.sign(t.iloc[:, 0]) == np.sign(t.iloc[:, 1])].mean(1))).items()]
print("  focal ratios moving the same way in both backgrounds:")
for a_ in agree_ratios:
    nm = a_.split(" @")[0]
    print(f"    {a_}" + (f"   [FRAGILE: {', '.join(frag[nm])} below the count floor]"
                         if frag.get(nm) else ""))
sep = Rt[Rt.separated]
print(f"  fully separated (no overlap between arms): "
      + (", ".join(f"{r.ratio}/{r.background}/{r.site}" for _, r in sep.iterrows()) or "none"))

print(f"\n[files written to {OUT}]")
for f_ in sorted(OUT.glob("*.csv")): print(f"  {f_.name}")
print("="*96)